In [17]:
import torch
import numpy as np
import torch.nn as nn
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

In [18]:
writer = SummaryWriter("runs/wine_classifier")

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
class WineDataset(Dataset):
    def __init__(self, X, Y):
        super(WineDataset, self).__init__()
        self.X = torch.Tensor(X)
        self.Y = torch.tensor(Y, dtype = torch.long)
    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.Y[idx]
        return x, y
    def __len__(self):
        return self.X.shape[0]

wine = datasets.load_wine()
X = wine.data
Y = wine.target
transform = StandardScaler()
X = transform.fit_transform(X)
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2)
train_dataset = WineDataset(x_train, y_train)
test_dataset = WineDataset(x_test, y_test)
train_dataloader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
test_dataloader = DataLoader(test_dataset, batch_size = 16, shuffle = True)

In [21]:
class Model(nn.Module):
    def __init__(self, input_size, hidden_units):
        super(Model, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_units)
        self.l2 = nn.Linear(hidden_units, 3)
        self.relu = nn.LeakyReLU(0.1)
    def forward(self, x):
        x = self.l1(x)
        x = self.relu(x)
        x = self.l2(x)
        return x

In [22]:
model = Model(x_train.shape[1], hidden_units = 16).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.005)
for epoch in range(15):
    pbar = tqdm(train_dataloader, desc = f"Epoch {epoch + 1}")
    avg_acc = []
    avg_loss = []
    for real, labels in pbar:
        real = real.to(device)
        labels = labels.to(device)
        out = model(real)
        loss = criterion(out, labels)
        acc = accuracy_score(labels.cpu().numpy(), torch.argmax(out, dim = 1).cpu().numpy())
        avg_loss.append(loss.item())
        avg_acc.append(acc)
        pbar.set_postfix(
            loss = f"{loss.item():.3f}", acc = f"{acc * 100:.2f}%")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    avg_acc = np.mean(avg_acc)
    avg_loss = np.mean(avg_loss)
    writer.add_scalar("Loss/Train", avg_loss, epoch + 1)
    writer.add_scalar("Accuracy/Train", avg_acc, epoch + 1)
writer.close()

Epoch 15: 100%|██████████████████████████████████████████████████| 9/9 [00:00<00:00, 69.44it/s, acc=78.57%, loss=0.875]


In [23]:
writer.add_graph(model, torch.Tensor(x_test).to(device))

In [24]:
test_x = torch.Tensor(x_test).to(device)
test_y = torch.tensor(y_test, dtype = torch.long).to(device)
test_out = model(test_x).detach()
acc = accuracy_score(test_y.cpu().numpy(), torch.argmax(test_out, dim = 1).cpu().numpy())
print(f'The test accuracy is {acc * 100}%')

The test accuracy is 88.88888888888889%


In [25]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [26]:
%tensorboard --logdir runs

In [27]:
!netstat -ano | findstr LISTENING | findstr :6006

  TCP    127.0.0.1:6006         0.0.0.0:0              LISTENING       20812


In [28]:
!taskkill /F /PID 20812

SUCCESS: The process with PID 20812 has been terminated.


In [29]:
!del /q %TMP%\.tensorboard-info\*